Total visa issuances by country, January 2021 - February 2026

In [1]:
## parse monthly pdfs from state department 

import re, io, zipfile
import pandas as pd
import pdfplumber

ZIP = r"C:\Users\CarolXu\Downloads\iv_fsc_pdfs_2021-02_to_2026.zip"

row_re = re.compile(r"^(?P<country>.*?)\s+(?P<cls>[A-Z][A-Z0-9]{1,3})\s+(?P<n>\d[\d,]*)$")
gt_re  = re.compile(r"Grand\s+Total\s+([\d,\s]+)$", re.I)   # some PDFs print it as "2 6,483"

all_rows, checks = [], []
with zipfile.ZipFile(ZIP) as z:
    for name in sorted(n for n in z.namelist() if n.endswith(".pdf")):
        date = pd.Timestamp(name[:7] + "-01")          # files are named YYYY-MM_FSC.pdf
        rows, gt = [], None
        with pdfplumber.open(io.BytesIO(z.read(name))) as pdf:
            for page in pdf.pages:
                for line in (page.extract_text() or "").splitlines():
                    line = re.sub(r"\s+", " ", line).strip()
                    if g := gt_re.search(line):
                        gt = int(re.sub(r"\D", "", g.group(1))); continue
                    if m := row_re.match(line):
                        rows.append((date, m["country"].strip(), m["cls"], int(m["n"].replace(",", ""))))
        all_rows += rows
        checks.append({"date": date, "parsed_total": sum(r[3] for r in rows), "pdf_grand_total": gt})

detail = pd.DataFrame(all_rows, columns=["date", "country", "visa_class", "issuances"])

# Make country names consistent across months
detail["country"] = (detail.country.str.replace("–", "-", regex=False)
                     .replace({"Cocos Islands": "Cocos (Keeling) Islands", "Saint Maarten": "Sint Maarten"}))

checks = pd.DataFrame(checks)
checks["match"] = checks.parsed_total.astype(int) == checks.pdf_grand_total.astype(int)
print(f"{checks.match.sum()} / {len(checks)} months match the PDF Grand Total")
print(detail.country.nunique(), "countries |", detail.visa_class.nunique(), "visa classes")

61 / 61 months match the PDF Grand Total
224 countries | 81 visa classes


In [2]:
## write csvs

detail["year"], detail["month"] = detail.date.dt.year, detail.date.dt.month

# CSV 1: monthly total (ALL COUNTRIES) plus each country's monthly total
by_country = detail.groupby(["date", "year", "month", "country"], as_index=False).issuances.sum()
overall = detail.groupby(["date", "year", "month"], as_index=False).issuances.sum().assign(country="ALL COUNTRIES")
totals = pd.concat([overall, by_country])
totals["_o"] = totals.country.ne("ALL COUNTRIES")
totals = totals.sort_values(["date", "_o", "country"]).drop(columns="_o")
totals["date"] = totals.date.dt.strftime("%Y-%m")
totals[["date", "year", "month", "country", "issuances"]].to_csv("iv_issuances_monthly_totals.csv", index=False)

# CSV 2: by country and visa class
out = detail.sort_values(["date", "country", "visa_class"]).copy()
out["date"] = out.date.dt.strftime("%Y-%m")
out[["date", "year", "month", "country", "visa_class", "issuances"]].to_csv("iv_issuances_by_country_visa_class.csv", index=False)

print(totals.shape, out.shape)

(10960, 5) (92566, 6)
